---
**Title: Numerical simulation for CP model in continuous racetrack**  
**Author: Kenuske Ohtake**  
**Date: 2026-7-26**  
**Updated: 2026-8-13**  
**Final review completed: 2026-9-14**

---

**Step 1**: Load equired packages

In [ ]:
using Random
using Distributions
using Plots
using Dates
using Format
using DataFrames
using CSV
using LinearAlgebra

**Step 2**: Define function `integral_operator`

Description:
- corresponds to integral operator for fixed point problem

Arguments:
- `Ex`: transport matrix - (N, N) array
- `W`: nominal wage raised to the power of σ - (N, 1) array
- `φ`: immobile workers - (N, 1) array
- `λ`: mobile workers - (N, 1) array
- `μ`: manufacturing ratio - scaler
- `σ`: elasticity of substitution - scaler

Returns:
- `output`: result of applying integral operator once - (N, 1) array

In [ ]:
function integral_operator(Ex, W, φ, λ, μ, σ)
    Y = (μ * ((W .^ (1.0 / σ)) .* λ)) .+ ((1.0 - μ) * φ) # nominal income Y
    G_ = Ex * (λ .* (W .^ ((1.0 / σ) - 1.0))) # price index G raised to the power of 1-σ but dx is omitted
    output = Ex * (Y ./ G_) # note: dx cancels out in both numerator and denominator
    return output
end

**Step 3**: Define function `instantaneous_equilibrium`

Description:
- calls `integral_operator` internally

Arguments:
 - `Ex`, `φ`, `λ`, `μ`, `σ`: same as in `integral_operator`
 - `W_cand`: candidate of fixed point W - (N, 1) array
 - `v`: adjustment coefficient of migration - scaler
 - `dx`: size of a line element - scaler
 - `ε`: convergence criterion - scaler

Returns:
- `rhs`: right-hand side of evolution equation
- `W`: nominal wage raised to the power of σ in instantaneous equilibrium

In [ ]:
function instantaneous_equilibrium(Ex, W_cand, φ, λ, μ, σ, v, dx, ε)

    # solve fixed point problem for instantaneous equilibrium
    W = copy(W_cand)
    while true
        W_old = copy(W)
        W = integral_operator(Ex, W_old, φ, λ, μ, σ)
        if maximum(abs.(W - W_old)) < ε
            break
        end
    end

    # price index
    G = ((dx * Ex) * (λ .* (W .^ ((1.0 / σ) - 1.0)))) .^ (1.0 / (1.0 - σ))

    # real wage
    ω = (W .^ (1.0 / σ)) .* (G .^ (-μ))

    # right-hand side of evolution equation
    rhs = v * ((ω .- (dot(ω, λ) * dx)) .* λ)

    return rhs, W
end

**Step 4**: Define function `cp_simulation`  
- calls `integral_operator` and `instantaneous_equilibrium` internally
- runs single simulation and save results (figures and csv files)

Argument:
- `parameters`: list consisting of control parameters σ and τ

In [ ]:
function cp_simulation(parameters)
  
    # == make several basic settings ==
 
    # determine random seed based on date
    SD = Dates.millisecond(now())
    Random.seed!(SD)
    
    # set control parameters 
    σ = parameters[1] # elasticity of substitution of manufactured goods
    τ = parameters[2] # manufacturing transportation cost parameter

    # define disnatnce function
    function dist(z)
        minimum([abs(z), 2.0 * π - abs(z)])
    end

    # set constants
    N = 256 # number of line elements
    dx = 2.0 * π / convert(Float64, N) # size of a line element
    dt = 0.01 # size of a time element
    μ = 0.6 # manufacturing ratio
    ε = 1e-10 # convergence criterion
    v = 1.0 # adjustment coefficient of migration

    # space-coodinate
    x = zeros(Float64, N, 1)
    for i in 1:N
        x[i] = -π + convert(Float64, i-1) * dx
    end
    
    # density of immobile workers ϕ
    φ = ones(Float64, (N, 1)) ./ (2.0 * π)
    
    # transport matrix
    Ex = zeros(Float64, (N, N))
    for i in 1:N
        for j in 1:N
            Ex[i, j] = exp(-τ * (σ - 1.0) * dist(x[i]-x[j]))
        end
    end
    
    # generate initial density of mobile workers λ
    Δλ = rand(Uniform(-0.0005, 0.0005), (N, 1)) # perturbation Δλ generated by random numbers
    Δλ = Δλ .- ((sum(Δλ) * dx) / (2.0 * π)) # adjust to ensure total amount of Δλ equals to 0
    λ = (ones(Float64, (N, 1)) ./ (2.0 * π)) + Δλ # λ = homogeneous state + Δλ

    # check non-negativity and total amount of λ
    if sum(Int.(sign.(λ))) != N
        error("Non-negativity is not satisfied!")
    else
        λ₀ = copy(λ) # save initial value of λ
        println("Initial total mobile workers is ", sum(λ₀) * dx)
    end
  
    # == main loop for time evolution ==
 
    # initialize time step
    t = 0

    # initialize candidate for nominal wage raised to the power of σ
    W = ones(Float64, (N, 1)) 

    # initialize the increment for density of mobile workers
    λ_increment = zeros(Float64, (N, 1))

    # iterate until statinary solution is obtained
    while true

        # save previous values
        λ_old = copy(λ)
        W_old = copy(W)
        
        # 1-st order Runge-Kutta (=Euler) method for replicator equation 
        k_1, W = instantaneous_equilibrium(Ex, W_old, φ, λ_old, μ, σ, v, dx, ε)
        λ_increment = dt * k_1
        λ = λ_old + λ_increment
        
        # check non-negativity
        if sum(Int.(sign.(λ))) != N
            error("Non-negativity is not satisfied!")
        end
        
        # check conservation of population
        if 1.0 - 1e-10 > sum(λ) * dx
            error("Conservation error!")
        elseif sum(λ) * dx > 1.0 + 1e-10
            error("Conservation error!")
        end

        # stationary condition
        if maximum(abs.(λ_increment)) < ε
            break
        end
    
        t += 1
    
    end
    
    # == obtain stationary states ==
   
    # nominal wage raised to the power of σ
    while true
        W_old= copy(W)
        W = integral_operator(Ex, W_old, φ, λ, μ, σ)
        if maximum(abs.(W - W_old)) < ε
            break
        end
    end
    
    # nominal wage
    w = W .^ (1.0 / σ)
    
    # price index
    G = ((dx * Ex) * (λ .* (w .^ (1.0 - σ)))) .^ (1.0 / (1.0 - σ))
    
    # real wage
    ω = w .* (G .^ (-μ))
    
    # print total amount of final λ
    println("Final total mobile workers is ", sum(λ) * dx)
   
    # == plot final results ==
   
    # periodic data for plotting
    DateTime = Dates.format(now(), "yyyymmddHHMMSS")
    x_plt = append!(reshape(x, N), π)
    λ₀_plt = append!(reshape(λ₀, N), λ₀[1])
    λ_plt = append!(reshape(λ, N), λ[1])
    w_plt = append!(reshape(w, N), w[1])
    G_plt = append!(reshape(G, N), G[1])
    ω_plt = append!(reshape(ω, N), ω[1])

    # plot final λ and λ₀
    filename_lambda = format("CP-{}-pop_tau{}sigma{}mu{}N{}t{}sd{}", DateTime, τ, σ, μ, N, t, SD)
    newfilename_lambda = replace(filename_lambda, "."=>"p")
    plot(x_plt, λ_plt, label="final", dpi=300)
    plot!(x_plt, λ₀_plt, title="mobile workers", label="initial", dpi=300)
    savefig(newfilename_lambda * ".png")

    # plot final ω
    filename_realwage = format("CP-{}-rw_tau{}sigma{}mu{}N{}t{}sd{}", DateTime, τ, σ, μ, N, t, SD)
    newfilename_realwage = replace(filename_realwage, "."=>"p")
    plot(x_plt, ω_plt, title="real wage", label="", dpi=300)
    savefig(newfilename_realwage * ".png")
 
    # == output and save data as csv ==

    # make dataframe
    df = DataFrame(coordinates = x_plt,
         lambda_initial = λ₀_plt,
         lambda_final = λ_plt,
         nominalwage = w_plt,
         priceindex = G_plt,
         realwage = ω_plt)
    
    # write into csv
    filename = format("CP-{}_tau{}sigma{}mu{}N{}t{}sd{}", DateTime, τ, σ, μ, N, t, SD)
    newfilename = replace(filename, "."=>"p")
    CSV.write(newfilename * ".csv", df)
end

**Step 5**: Run simulations  
- run `n_simulations` times simulations for each parameter pair $(\sigma, \tau)$

In [ ]:
# == settings ==

sigmas = [5.0] # range of σ
taus = [1.8, 1.9, 2.0] # range of τ
n_simulations = 5 # number of simulations for each parameter

# == run multiple simulations ==

for sigma in sigmas
    for tau in taus
        control_parameters = [sigma, tau]
        for experiments in 1:n_simulations
            cp_simulation(control_parameters)
        end
    end
end